# Tutorial: Direct vs Programmatic Tool Calling — Inventory Replenishment

Compare two Responses API orchestration modes on the same deterministic inventory task. The notebook starts offline, requires an explicit opt-in for live API calls, and treats quality as a hard gate before comparing estimated cost.


## Audience, prerequisites, and learning goals

This tutorial is for developers who already understand basic function calling and want to evaluate Programmatic Tool Calling with evidence instead of intuition.

Prerequisites:

- Python 3.11 or newer and `uv`
- An OpenAI API key only for the optional live section
- Familiarity with Responses API input and output items

By the end, you will be able to:

1. Build equivalent Direct and Programmatic tool surfaces.
2. Preserve `call_id` and program `caller` linkage during stateless continuation.
3. Compare tokens, estimated cost, calls, turns, and latency only after quality gates pass.
4. Inspect the generated JavaScript and the final `program_output` separately from the assistant message.


## Outline

1. Configure a safe offline-first run.
2. Inspect deterministic detailed inventory fixtures and the answer oracle.
3. Compare tool definitions and orchestration prompts.
4. Run one opt-in live comparison and inspect its timeline.
5. Optionally repeat each arm three times.
6. Interpret the result, review pitfalls, and complete an exercise.


## 1. Setup

The first cell locates the project root and imports the reusable benchmark package. It does not load or print an API key.


In [1]:
from __future__ import annotations

import json
import os
import sys
import uuid
from pathlib import Path

from IPython.display import Markdown, display

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src" / "ptc_benchmark").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src" / "ptc_benchmark").exists():
    raise RuntimeError("Run this notebook from the project root or notebooks directory.")

sys.path.insert(0, str(PROJECT_ROOT / "src"))

from ptc_benchmark.config import configured_model, load_local_environment, require_api_key
from ptc_benchmark.inventory_evaluation import evaluate_inventory_run
from ptc_benchmark.inventory import build_inventory_dataset
from ptc_benchmark.pricing import estimate_run_cost, load_pricing_catalog
from ptc_benchmark.reporting import append_jsonl, comparison_rows, markdown_table, request_timeline
from ptc_benchmark.runner import InventoryRunner, RunConfig


### Safe execution controls

Live and repeated runs are disabled by default. `medium` uses 10 SKUs and therefore requires 30 function calls per arm. Use `small` for the lowest-cost smoke run and `large` only after the workflow is stable.


In [2]:
RUN_LIVE = True # Set to True to enable live API calls and associated cost.
RUN_REPEATED_COMPARISON = True # Set to True for repeated runs; requires RUN_LIVE = True.
REPEATS = 3

SCALE = "medium"  # small=3 SKUs, medium=10, large=30
MODEL = configured_model("gpt-5.6")
REASONING_EFFORT = "medium"

PRICING_PATH = Path(
    os.getenv(
        "OPENAI_PRICING_PATH",
        PROJECT_ROOT / "pricing" / "openai_pricing_2026-08-06.json",
    )
)
RESULTS_PATH = PROJECT_ROOT / "results" / "inventory_runs.jsonl"

print(
    {
        "RUN_LIVE": RUN_LIVE,
        "RUN_REPEATED_COMPARISON": RUN_REPEATED_COMPARISON,
        "scale": SCALE,
        "model": MODEL,
        "reasoning_effort": REASONING_EFFORT,
    }
)


{'RUN_LIVE': True, 'RUN_REPEATED_COMPARISON': True, 'scale': 'medium', 'model': 'gpt-5.6', 'reasoning_effort': 'medium'}


## 2. Controlled experiment

Both arms receive the same model, reasoning effort, task requirements, SKU list, detailed fixture data, output contract, and final-answer requirements. Direct Tool Calling is allowed to issue parallel function calls. The intended difference is where intermediate results are processed:

- **Direct:** detailed tool outputs return to model context for calculation.
- **Programmatic:** generated JavaScript calls tools and reduces their outputs inside the hosted runtime.

Each arm receives an isolated `prompt_cache_key`, reused only across turns in that one run. This prevents one arm or repetition from warming another arm's cache.


## 3. Deterministic fixtures and oracle

Each SKU has three warehouse rows, seven daily forecast rows, and three shipment rows. Only scheduled shipments arriving within seven days count as inbound units. The local oracle applies the same published formula without using a model.


In [3]:
dataset = build_inventory_dataset(SCALE)
expected_plan = dataset.expected_plan()

display(Markdown(f"**SKUs:** {len(dataset.skus)}  \n**Expected tool calls per arm:** {len(dataset.skus) * 3}  \n**Safety stock:** {dataset.safety_stock}"))
display(Markdown("### Oracle result\n```json\n" + json.dumps(expected_plan, indent=2) + "\n```"))


SKUs: 10 
 Expected tool calls per arm: 30 
 Safety stock: 5

Oracle result ¶ { 
 "recommendations" : [ 
 { 
 "sku" : "sku-010" , 
 "available_units" : 15 , 
 "forecast_units" : 24 , 
 "inbound_units" : 0 , 
 "reorder_units" : 14 
 }, 
 { 
 "sku" : "sku-007" , 
 "available_units" : 15 , 
 "forecast_units" : 22 , 
 "inbound_units" : 0 , 
 "reorder_units" : 12 
 }, 
 { 
 "sku" : "sku-004" , 
 "available_units" : 15 , 
 "forecast_units" : 20 , 
 "inbound_units" : 0 , 
 "reorder_units" : 10 
 }, 
 { 
 "sku" : "sku-001" , 
 "available_units" : 15 , 
 "forecast_units" : 18 , 
 "inbound_units" : 0 , 
 "reorder_units" : 8 
 } 
 ], 
 "total_reorder_units" : 44 
 }

In [4]:
sample_sku = dataset.skus[0]
sample_payloads = {
    name: dataset.execute(name, {"sku": sample_sku})
    for name in ("get_inventory", "get_weekly_demand", "get_inbound_shipments")
}
display(Markdown(f"### Detailed tool payloads for `{sample_sku}`"))
display(Markdown("```json\n" + json.dumps(sample_payloads, indent=2) + "\n```"))


Detailed tool payloads for sku-001 ¶

{ 
 "get_inventory" : { 
 "sku" : "sku-001" , 
 "warehouses" : [ 
 { 
 "warehouse_id" : "wh-1" , 
 "on_hand_units" : 4 , 
 "reserved_units" : 0 
 }, 
 { 
 "warehouse_id" : "wh-2" , 
 "on_hand_units" : 6 , 
 "reserved_units" : 1 
 }, 
 { 
 "warehouse_id" : "wh-3" , 
 "on_hand_units" : 8 , 
 "reserved_units" : 2 
 } 
 ] 
 }, 
 "get_weekly_demand" : { 
 "sku" : "sku-001" , 
 "daily_forecast" : [ 
 { 
 "date" : "2026-08-11" , 
 "units" : 1 
 }, 
 { 
 "date" : "2026-08-12" , 
 "units" : 2 
 }, 
 { 
 "date" : "2026-08-13" , 
 "units" : 3 
 }, 
 { 
 "date" : "2026-08-14" , 
 "units" : 4 
 }, 
 { 
 "date" : "2026-08-15" , 
 "units" : 5 
 }, 
 { 
 "date" : "2026-08-16" , 
 "units" : 1 
 }, 
 { 
 "date" : "2026-08-17" , 
 "units" : 2 
 } 
 ] 
 }, 
 "get_inbound_shipments" : { 
 "sku" : "sku-001" , 
 "shipments" : [ 
 { 
 "shipment_id" : "ship-001-a" , 
 "eta_date" : "2026-08-12" , 
 "units" : 3 , 
 "status" : "delayed" 
 }, 
 { 
 "shipment_id" : "ship-001-b" , 
 "eta_date" : "2026-08-20" , 
 "units" : 8 , 
 "status" : "scheduled" 
 }, 
 { 
 "shipment_id" : "ship-001-c" , 
 "eta_date" : "2026-08-09" , 
 "units" : 2 , 
 "status" : "arrived" 
 } 
 ] 
 } 
 }

## 4. Equivalent tool surfaces

The three function schemas are identical except for `allowed_callers`. The Programmatic arm also exposes the hosted `programmatic_tool_calling` tool. Predictable `output_schema` definitions let generated JavaScript safely read structured fields.


In [5]:
tool_summary = []
for arm in ("direct", "programmatic"):
    for tool in dataset.tool_definitions(arm):
        tool_summary.append(
            {
                "arm": arm,
                "type": tool["type"],
                "name": tool.get("name", "hosted runtime"),
                "allowed_callers": tool.get("allowed_callers", []),
                "has_output_schema": "output_schema" in tool,
            }
        )
display(Markdown(markdown_table(tool_summary)))


arm,type,name,allowed_callers,has_output_schema
direct,function,get_inventory,['direct'],True
direct,function,get_weekly_demand,['direct'],True
direct,function,get_inbound_shipments,['direct'],True
programmatic,function,get_inventory,['programmatic'],True
programmatic,function,get_weekly_demand,['programmatic'],True
programmatic,function,get_inbound_shipments,['programmatic'],True
programmatic,programmatic_tool_calling,hosted runtime,[],False


## 5. Arm-specific orchestration

The task contract remains the same. Only the orchestration section changes: Direct calls functions itself, while Programmatic must create all promises, use `Promise.all`, calculate inside JavaScript, and emit one reduced JSON object.


In [6]:
for arm in ("direct", "programmatic"):
    instructions, user_prompt = dataset.prompt(arm)
    orchestration = instructions.split("<tool_orchestration>", 1)[1].split("</tool_orchestration>", 1)[0].strip()
    display(Markdown(f"### {arm.title()} orchestration\n```text\n{orchestration}\n```"))
print(f"Shared user prompt: {user_prompt}")


Direct orchestration ¶ Use Direct Tool Calling. Call the functions directly and issue independent calls in
parallel when possible. Use the returned tool data to calculate the result. Do not
write or execute a programmatic_tool_calling program.

Programmatic orchestration ¶ Use Programmatic Tool Calling for the complete lookup and calculation stage. Create
all tool-call promises before awaiting them and resolve them with Promise.all. Perform
all filtering, summation, sorting, and reduction inside the generated JavaScript.
Emit exactly the required JSON object from the program with text(JSON.stringify(result)).
After the program completes, write the required RESULT_JSON and EXPLANATION final message.
Do not call the inventory functions directly.

Shared user prompt: Which products should we reorder this week, and in what quantities?


## 6. Offline quality gates

Before spending API tokens, verify that fixtures are complete and the oracle result is internally consistent. Live runs add five hard gates: structured result, final `RESULT_JSON`, evidence-complete explanation, exact tool coverage, and correct caller linkage.


In [7]:
offline_checks = {
    "sku_count_matches_scale": len(dataset.skus) in {3, 10, 30},
    "three_payloads_per_sku": all(
        sku in dataset.inventory and sku in dataset.demand and sku in dataset.inbound
        for sku in dataset.skus
    ),
    "only_positive_recommendations": all(
        row["reorder_units"] > 0 for row in expected_plan["recommendations"]
    ),
    "total_matches_rows": expected_plan["total_reorder_units"]
    == sum(row["reorder_units"] for row in expected_plan["recommendations"]),
}
assert all(offline_checks.values()), offline_checks
display(Markdown(markdown_table([offline_checks])))


sku_count_matches_scale,three_payloads_per_sku,only_positive_recommendations,total_matches_rows
True,True,True,True


## 7. Optional live Direct vs Programmatic comparison

This cell does nothing unless `RUN_LIVE = True`. It loads `.env.local`, checks for an API key without displaying it, runs both arms once, evaluates quality, estimates cost from the dated pricing snapshot, and appends raw data to a gitignored JSONL file.


In [8]:
live_results = {}

if not RUN_LIVE:
    print("Live comparison skipped. Set RUN_LIVE = True to opt in to API usage and cost.")
else:
    from openai import OpenAI

    load_local_environment(PROJECT_ROOT)
    require_api_key()
    pricing = load_pricing_catalog(PRICING_PATH)
    runner = InventoryRunner(OpenAI())
    expected_tool_calls = len(dataset.skus) * 3
    run_config = RunConfig(
        model=MODEL,
        reasoning_effort=REASONING_EFFORT,
        max_requests=expected_tool_calls + 2,
    )
    comparison_id = f"inventory-{uuid.uuid4().hex[:10]}"

    for arm in ("direct", "programmatic"):
        run = runner.run(
            arm=arm,
            dataset=dataset,
            config=run_config,
            run_id=comparison_id,
        )
        evaluation = evaluate_inventory_run(run, dataset)
        cost = estimate_run_cost(run, pricing)
        live_results[arm] = (run, evaluation, cost)
        append_jsonl(RESULTS_PATH, run, evaluation, cost)

    display(Markdown(markdown_table(comparison_rows(live_results.values()))))
    for arm, (_, evaluation, _) in live_results.items():
        if not evaluation.passed:
            print(f"{arm} failed quality gates: {evaluation.failures}")


arm,passed,requests,tool_calls,input_tokens,cached_tokens,cache_write_tokens,output_tokens,reasoning_tokens,estimated_cost_usd,end_to_end_seconds
direct,True,2,30,4577,0,4047,1063,260,0.059834,13.651
programmatic,True,2,30,3067,1185,1745,808,154,0.036424,15.309


## 8. Inspect request timelines and generated JavaScript

Token and latency metrics are shown per request as well as end to end. The Programmatic trace should contain `program`, program-owned `function_call` items, a `program_output`, and a final `message`. A cost result is eligible to win only when every quality gate passes.


In [9]:
if not live_results:
    print("No live traces to display.")
else:
    for arm, (run, evaluation, cost) in live_results.items():
        display(Markdown(f"### {arm.title()} request timeline"))
        display(Markdown(markdown_table(request_timeline(run))))
        display(Markdown(f"**Quality passed:** `{evaluation.passed}`  \n**Estimated cost:** `${cost.total_cost:.6f}`  \n**End-to-end latency:** `{run.total_latency_seconds:.3f}s`"))

    programmatic_run = live_results["programmatic"][0]
    if programmatic_run.generated_programs:
        display(Markdown("### Generated JavaScript\n```javascript\n" + programmatic_run.generated_programs[-1] + "\n```"))
    if programmatic_run.program_outputs:
        display(Markdown("### Program output\n```json\n" + json.dumps(programmatic_run.program_outputs[-1], indent=2) + "\n```"))


Direct request timeline ¶

request,output_types,input_tokens,cached_tokens,cache_write_tokens,output_tokens,latency_seconds
1,"reasoning, function_call, function_call, function_call, function_call, function_call, function_call, function_call, function_call, function_call, function_call, function_call, function_call, function_call, function_call, function_call, function_call, function_call, function_call, function_call, function_call, function_call, function_call, function_call, function_call, function_call, function_call, function_call, function_call, function_call, function_call",527,0,0,577,7.012
2,"reasoning, message",4050,0,4047,486,6.636


Quality passed: True 
 Estimated cost: $0.059834 
 End-to-end latency: 13.651s

Programmatic request timeline ¶

request,output_types,input_tokens,cached_tokens,cache_write_tokens,output_tokens,latency_seconds
1,"reasoning, program, function_call, function_call, function_call, function_call, function_call, function_call, function_call, function_call, function_call, function_call, function_call, function_call, function_call, function_call, function_call, function_call, function_call, function_call, function_call, function_call, function_call, function_call, function_call, function_call, function_call, function_call, function_call, function_call, function_call, function_call",1188,0,1054,556,11.129
2,"program_output, reasoning, message",1879,1185,691,252,4.179


Quality passed: True 
 Estimated cost: $0.036424 
 End-to-end latency: 15.309s

Generated JavaScript ¶ const skus = Array . from ({ length : 10 }, ( _ , i ) => `sku- ${ String ( i + 1 ). padStart ( 3 , "0" ) } ` ); 

 const inventoryPromises = skus . map ( sku => tools . get_inventory ({ sku })); 
 const demandPromises = skus . map ( sku => tools . get_weekly_demand ({ sku })); 
 const inboundPromises = skus . map ( sku => tools . get_inbound_shipments ({ sku })); 
 const allPromises = [... inventoryPromises , ... demandPromises , ... inboundPromises ]; 

 const allResults = await Promise . all ( allPromises ); 
 const inventories = allResults . slice ( 0 , 10 ); 
 const demands = allResults . slice ( 10 , 20 ); 
 const inbounds = allResults . slice ( 20 , 30 ); 

 const recommendations = skus . map (( sku , i ) => { 
 const available_units = inventories [ i ]. warehouses . reduce ( 
 ( sum , row ) => sum + row . on_hand_units - row . reserved_units , 0 
 ); 
 const forecast_units = demands [ i ]. daily_forecast . reduce ( 
 ( sum , row ) => sum + row . units , 0 
 ); 
 const inbound_units = inbounds [ i ]. shipments 
 . filter ( row => row . status === "scheduled" && row . eta_date <= "2026-08-17" ) 
 . reduce (( sum , row ) => sum + row . units , 0 ); 
 const reorder_units = Math . max ( 
 forecast_units + 5 - available_units - inbound_units , 0 
 ); 
 return { sku , available_units , forecast_units , inbound_units , reorder_units }; 
 }). filter ( row => row . reorder_units > 0 ) 
 . sort (( a , b ) => b . reorder_units - a . reorder_units || a . sku . localeCompare ( b . sku )); 

 const result = { 
 recommendations , 
 total_reorder_units : recommendations . reduce (( sum , row ) => sum + row . reorder_units , 0 ) 
 }; 

 text ( JSON . stringify ( result ));

Program output ¶ { 
 "recommendations" : [ 
 { 
 "sku" : "sku-010" , 
 "available_units" : 15 , 
 "forecast_units" : 24 , 
 "inbound_units" : 0 , 
 "reorder_units" : 14 
 }, 
 { 
 "sku" : "sku-007" , 
 "available_units" : 15 , 
 "forecast_units" : 22 , 
 "inbound_units" : 0 , 
 "reorder_units" : 12 
 }, 
 { 
 "sku" : "sku-004" , 
 "available_units" : 15 , 
 "forecast_units" : 20 , 
 "inbound_units" : 0 , 
 "reorder_units" : 10 
 }, 
 { 
 "sku" : "sku-001" , 
 "available_units" : 15 , 
 "forecast_units" : 18 , 
 "inbound_units" : 0 , 
 "reorder_units" : 8 
 } 
 ], 
 "total_reorder_units" : 44 
 }

## 9. Optional three-run comparison

A single run is useful for understanding the item flow but not for generalizing model behavior or service latency. Enable both live flags to run three isolated repetitions per arm. Every repetition receives a distinct cache key; quality pass rate is reported alongside cost and latency.


In [10]:
repeated_rows = []

if not RUN_REPEATED_COMPARISON:
    print("Repeated comparison skipped. This is the safe default.")
elif not RUN_LIVE:
    raise RuntimeError("RUN_REPEATED_COMPARISON requires RUN_LIVE = True.")
else:
    from openai import OpenAI

    load_local_environment(PROJECT_ROOT)
    require_api_key()
    pricing = load_pricing_catalog(PRICING_PATH)
    runner = InventoryRunner(OpenAI())
    expected_tool_calls = len(dataset.skus) * 3
    run_config = RunConfig(
        model=MODEL,
        reasoning_effort=REASONING_EFFORT,
        max_requests=expected_tool_calls + 2,
    )

    for repeat_index in range(REPEATS):
        repeat_id = f"inventory-repeat-{repeat_index + 1}-{uuid.uuid4().hex[:8]}"
        for arm in ("direct", "programmatic"):
            run = runner.run(arm=arm, dataset=dataset, config=run_config, run_id=repeat_id)
            evaluation = evaluate_inventory_run(run, dataset)
            cost = estimate_run_cost(run, pricing)
            append_jsonl(RESULTS_PATH, run, evaluation, cost)
            repeated_rows.extend(comparison_rows([(run, evaluation, cost)]))

    display(Markdown(markdown_table(repeated_rows)))


arm,passed,requests,tool_calls,input_tokens,cached_tokens,cache_write_tokens,output_tokens,reasoning_tokens,estimated_cost_usd,end_to_end_seconds
direct,True,2,30,4605,0,4075,1059,269,0.059889,12.838
programmatic,True,2,30,2998,1185,1676,733,111,0.033742,14.465
direct,True,2,30,4577,0,4047,1094,304,0.060764,14.658
programmatic,True,2,30,3148,1185,1826,888,209,0.03933,18.644
direct,True,2,30,4582,0,4052,1037,246,0.059085,12.241
programmatic,True,12,30,3412,1185,2090,1154,470,0.04896,30.991


## 10. How to interpret the result

Use this order:

1. **Quality:** Exclude any run that fails a hard gate.
2. **Context:** Compare input, cached, and cache-write tokens across the complete loop.
3. **Generation:** Compare output and reasoning tokens, including generated JavaScript.
4. **Orchestration:** Compare Responses requests and function calls.
5. **Latency:** Compare request latency and task end-to-end latency separately.
6. **Estimated cost:** Apply the same dated price snapshot only after the earlier checks.

Programmatic Tool Calling may reduce intermediate context for larger detailed payloads, but generated code and additional model work also consume tokens. The measured crossover matters more than a universal claim.


## Exercise

Run the live comparison first with `SCALE = "small"`, then with `SCALE = "medium"`.

Before running, write down your hypothesis:

- Which arm will use fewer total input tokens?
- Will the lower-token arm also have lower end-to-end latency?
- Does either arm fail a quality gate as the number of calls increases?

Use the helper below to calculate deltas only when both arms pass.


In [11]:
def quality_adjusted_delta(rows, metric):
    # Return programmatic minus direct only after both arms pass.
    by_arm = {row["arm"]: row for row in rows}
    if set(by_arm) != {"direct", "programmatic"}:
        raise ValueError("Expected one direct and one programmatic row")
    if not all(by_arm[arm]["passed"] for arm in by_arm):
        return None
    return by_arm["programmatic"][metric] - by_arm["direct"][metric]


if live_results:
    rows = comparison_rows(live_results.values())
    print("Input-token delta:", quality_adjusted_delta(rows, "input_tokens"))
    print("Estimated-cost delta:", quality_adjusted_delta(rows, "estimated_cost_usd"))
else:
    print("Run the optional live comparison to complete the exercise.")


Input-token delta: -1510
Estimated-cost delta: -0.02341


## Pitfalls and extensions

Common pitfalls:

- Do not compare PTC with an artificially serial Direct baseline; this notebook permits Direct parallel calls.
- Do not declare the cheapest failed run a winner.
- Do not drop `caller` when returning a program-owned `function_call_output`; the hosted runtime needs it to resume the correct program.
- Do not compare arms with shared cache state or different models, reasoning settings, fixtures, or output requirements.
- Do not treat request latency as task end-to-end latency.

Extensions:

- Add a compact scalar payload profile and measure the payload-size crossover.
- Add fixed local tool latency to separate orchestration latency from model latency.
- Add the incident-investigation scenario, where each result requires fresh semantic judgment and Direct Tool Calling may be the stronger route.
